In [1]:
!pip install -q "transformers>=4.45,<5" torch numpy pandas scikit-learn


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.0 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
import torch
import numpy as np

from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

MODEL_NAME = "distilbert/distilroberta-base"
NUM_LABELS = 41

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", DEVICE)


Using device: cuda


In [3]:
def initialize_model(
    model_name=MODEL_NAME,
    num_labels=NUM_LABELS
):
    tokenizer = AutoTokenizer.from_pretrained(
        model_name
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_labels,
        problem_type="multi_label_classification"
    )

    model = model.to(DEVICE)

    return model, tokenizer


model, tokenizer = initialize_model()

print("Model initialized successfully!")
print("Number of labels:", model.config.num_labels)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/331M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at distilbert/distilroberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model initialized successfully!
Number of labels: 41


In [4]:
class ContractDataset(Dataset):

    def __init__(self, texts, labels, tokenizer, max_length=256):
        assert len(texts) == len(labels)

        labels = np.asarray(labels, dtype=np.float32)

        assert labels.ndim == 2
        assert labels.shape[1] == NUM_LABELS

        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            max_length=max_length
        )

        self.labels = torch.tensor(
            labels,
            dtype=torch.float32
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        item = {
            key: torch.tensor(values[idx])
            for key, values in self.encodings.items()
        }

        item["labels"] = self.labels[idx]

        return item


In [6]:
# ===== Rebuild the team's CUAD train/val/test splits (Aditi's processing) =====
import os, json, re
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer

REPO = "/content/team_repo"
if not os.path.exists(REPO):
    os.system(f"git clone -q --depth 1 https://github.com/Break-Through-Tech/accenture-1k-contract-review-challenge.git {REPO}")
DATA = f"{REPO}/data"

with open(f"{DATA}/train_separate_questions.json") as f:
    train_data = json.load(f)
with open(f"{DATA}/test.json") as f:
    test_data = json.load(f)

def get_category(qa):
    return re.sub(r"_\d+$", "", qa["id"].split("__")[-1])

def extract_annotations(contract):
    anns = []
    for p in contract["paragraphs"]:
        for qa in p["qas"]:
            cat = get_category(qa)
            for a in qa["answers"]:
                anns.append({"category": cat,
                             "answer_start": a["answer_start"],
                             "answer_end": a["answer_start"] + len(a["text"])})
    return anns

def create_chunks(context, max_length=512, overlap=64):
    content_length = max_length - 2          # room for RoBERTa special tokens
    step = content_length - overlap
    enc = tokenizer(context, add_special_tokens=False,
                    return_offsets_mapping=True, truncation=False)
    ids, offsets = enc["input_ids"], enc["offset_mapping"]
    chunks = []
    for chunk_id, start in enumerate(range(0, len(ids), step)):
        end = min(start + content_length, len(ids))
        cs, ce = offsets[start][0], offsets[end - 1][1]
        chunks.append({"chunk_id": chunk_id, "chunk_start": cs, "chunk_end": ce,
                       "chunk_text": context[cs:ce],
                       "token_count": (end - start) + 2, "labels": []})
        if end == len(ids):
            break
    return chunks

def label_chunks(chunks, anns):
    for a in anns:
        s, e, cat = a["answer_start"], a["answer_end"], a["category"]
        full = [c for c in chunks if s >= c["chunk_start"] and e <= c["chunk_end"]]
        if full:
            targets = full
        else:
            overl = [(max(0, min(e, c["chunk_end"]) - max(s, c["chunk_start"])), c) for c in chunks]
            overl = [x for x in overl if x[0] > 0]
            if not overl:
                continue
            targets = [max(overl, key=lambda x: x[0])[1]]
        for c in targets:
            if cat not in c["labels"]:
                c["labels"].append(cat)
    return chunks

def process_contracts(contracts, source):
    rows = []
    for i, contract in enumerate(contracts):
        cid = contract.get("title", f"{source}_contract_{i}")
        context = contract["paragraphs"][0]["context"]
        for c in label_chunks(create_chunks(context), extract_annotations(contract)):
            rows.append({"contract_id": cid, "source": source, **c})
    return rows

print("Chunking contracts (takes a minute or two)...")
chunked_df = pd.DataFrame(process_contracts(train_data["data"], "original_train")
                          + process_contracts(test_data["data"], "original_test"))

# Same contract-level 80/10/10 split as Aditi's notebook (seed 42)
contract_ids = np.array(chunked_df["contract_id"].unique(), dtype=object)
rng = np.random.default_rng(42)
rng.shuffle(contract_ids)
n = len(contract_ids)
n_train, n_val = int(n * 0.8), int(n * 0.1)
train_ids = set(contract_ids[:n_train])
val_ids = set(contract_ids[n_train:n_train + n_val])

train_chunks = chunked_df[chunked_df["contract_id"].isin(train_ids)].reset_index(drop=True)
val_chunks = chunked_df[chunked_df["contract_id"].isin(val_ids)].reset_index(drop=True)
assert not (train_ids & val_ids), "train and val share a contract"

# Multi-hot labels in a fixed, sorted category order
label_names = sorted({c for labs in chunked_df["labels"] for c in labs})
mlb = MultiLabelBinarizer(classes=label_names)
mlb.fit([[]])
train_labels = mlb.transform(train_chunks["labels"]).astype(np.float32)
val_labels = mlb.transform(val_chunks["labels"]).astype(np.float32)
train_texts = train_chunks["chunk_text"].tolist()
val_texts = val_chunks["chunk_text"].tolist()

print("Contracts:", n, "| train:", len(train_ids), "| val:", len(val_ids))
print("Train chunks:", len(train_texts), "| Val chunks:", len(val_texts))
print("Categories found:", len(label_names), "(should be 41)")
print("Train chunks with no clause:", round(float((train_labels.sum(1) == 0).mean()), 3))

Token indices sequence length is longer than the specified maximum sequence length for this model (23248 > 512). Running this sequence through the model will result in indexing errors


Chunking contracts (takes a minute or two)...
Contracts: 510 | train: 408 | val: 51
Train chunks: 11500 | Val chunks: 1998
Categories found: 41 (should be 41)
Train chunks with no clause: 0.618


In [7]:
train_ds = ContractDataset(train_texts, train_labels, tokenizer, max_length=512)
val_ds   = ContractDataset(val_texts, val_labels, tokenizer, max_length=512)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=32)

print("Train batches:", len(train_loader), "| Val batches:", len(val_loader))

Train batches: 719 | Val batches: 63


In [8]:
import random
import torch.nn as nn
from sklearn.metrics import f1_score, precision_score, recall_score

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

In [9]:
def analyze_labels(train_labels, label_names):
    n = len(train_labels)
    pos = train_labels.sum(axis=0)
    df = pd.DataFrame({
        "label": label_names,
        "positives": pos.astype(int),
        "freq_%": 100 * pos / n,
    })
    df["bucket"] = pd.cut(df["freq_%"], bins=[-1, 1, 5, 100],
                          labels=["rare", "medium", "common"])
    return df

label_df = analyze_labels(train_labels, label_names)
print(label_df.sort_values("positives").to_string())
print(label_df["bucket"].value_counts())

                                 label  positives    freq_%  bucket
30                  Price Restrictions         20  0.173913    rare
38   Unlimited/All-You-Can-Eat-License         22  0.191304    rare
34                  Source Code Escrow         25  0.217391    rare
21                 Most Favored Nation         25  0.217391    rare
22             No-Solicit Of Customers         33  0.286957    rare
36             Third Party Beneficiary         33  0.286957    rare
25                   Non-Disparagement         38  0.330435    rare
1           Affiliate License-Licensor         39  0.339130    rare
23             No-Solicit Of Employees         56  0.486957    rare
17                  Joint Ip Ownership         63  0.547826    rare
0           Affiliate License-Licensee         64  0.556522    rare
7    Competitive Restriction Exception         80  0.695652    rare
16    Irrevocable Or Perpetual License         82  0.713043    rare
19                  Liquidated Damages         8

In [10]:
def compute_pos_weight(train_labels, mode="sqrt", cap=50.0):
    pos = train_labels.sum(axis=0)
    neg = len(train_labels) - pos
    w = neg / np.clip(pos, 1, None)
    if mode == "sqrt":
        w = np.sqrt(w)
    w = np.clip(w, 1.0, cap)
    return torch.tensor(w, dtype=torch.float32)

pw_preview = compute_pos_weight(train_labels, mode="sqrt")
print(pd.DataFrame({"label": label_names, "pos_weight": pw_preview.numpy().round(2)})
        .sort_values("pos_weight", ascending=False).to_string())

                                 label  pos_weight
30                  Price Restrictions   23.959999
38   Unlimited/All-You-Can-Eat-License   22.840000
21                 Most Favored Nation   21.420000
34                  Source Code Escrow   21.420000
36             Third Party Beneficiary   18.639999
22             No-Solicit Of Customers   18.639999
25                   Non-Disparagement   17.370001
1           Affiliate License-Licensor   17.139999
23             No-Solicit Of Employees   14.300000
17                  Joint Ip Ownership   13.470000
0           Affiliate License-Licensee   13.370000
7    Competitive Restriction Exception   11.950000
16    Irrevocable Or Perpetual License   11.800000
19                  Liquidated Damages   11.660000
40                   Warranty Duration   10.960000
27  Notice Period To Terminate Renewal   10.900000
37                  Uncapped Liability   10.620000
39                  Volume Restriction   10.180000
8                  Covenant Not

In [11]:
def train_model_weighted(model, train_loader, epochs=3,
                         learning_rate=2e-5, pos_weight=None):
    optimizer = AdamW(model.parameters(), lr=learning_rate)
    loss_fn = nn.BCEWithLogitsLoss(
        pos_weight=pos_weight.to(DEVICE) if pos_weight is not None else None
    )
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in train_loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            labels = batch.pop("labels")
            optimizer.zero_grad()
            logits = model(**batch).logits
            loss = loss_fn(logits, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(train_loader):.4f}")
    return model

In [12]:
def get_probs(model, loader):
    model.eval()
    all_p, all_y = [], []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            y = batch.pop("labels")
            all_p.append(torch.sigmoid(model(**batch).logits).cpu())
            all_y.append(y.cpu())
    return torch.cat(all_p).numpy(), torch.cat(all_y).numpy()

def evaluate(probs, y, label_df, threshold=0.5):
    preds = (probs >= threshold).astype(int)
    per = label_df.copy()
    per["precision"] = precision_score(y, preds, average=None, zero_division=0)
    per["recall"]    = recall_score(y, preds, average=None, zero_division=0)
    per["f1"]        = f1_score(y, preds, average=None, zero_division=0)
    summary = {
        "micro_f1": f1_score(y, preds, average="micro", zero_division=0),
        "macro_f1": f1_score(y, preds, average="macro", zero_division=0),
    }
    by_bucket = per.groupby("bucket", observed=True)[["precision", "recall", "f1"]].mean()
    return summary, by_bucket, per

In [13]:
configs = {
    "standard_bce":  None,
    "weighted_sqrt": compute_pos_weight(train_labels, mode="sqrt", cap=50),
    # "weighted_raw": compute_pos_weight(train_labels, mode="raw", cap=100),  # add later if time
}

results = {}
for name, pw in configs.items():
    print(f"\n=== {name} ===")
    set_seed(42)
    model, tokenizer = initialize_model()     # fresh model each run
    model = train_model_weighted(model, train_loader, epochs=2,
                                 learning_rate=2e-5, pos_weight=pw)
    probs, y = get_probs(model, val_loader)
    results[name] = evaluate(probs, y, label_df)

print("\n=== Overall ===")
print(pd.DataFrame({k: v[0] for k, v in results.items()}).T.round(3))
for k, v in results.items():
    print(f"\n{k} by bucket:\n", v[1].round(3))


=== standard_bce ===


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at distilbert/distilroberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/2 - Loss: 0.1170
Epoch 2/2 - Loss: 0.0712

=== weighted_sqrt ===


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at distilbert/distilroberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/2 - Loss: 0.3486
Epoch 2/2 - Loss: 0.2160

=== Overall ===
               micro_f1  macro_f1
standard_bce      0.256      0.07
weighted_sqrt     0.491      0.37

standard_bce by bucket:
         precision  recall    f1
bucket                         
rare        0.000   0.000  0.00
medium      0.125   0.137  0.13

weighted_sqrt by bucket:
         precision  recall     f1
bucket                          
rare        0.203   0.399  0.223
medium      0.419   0.686  0.498


In [14]:
# Re-bucket: rare <1%, medium 1–3%, frequent >=3% of training chunks
new_bucket = pd.cut(label_df["freq_%"], bins=[-1, 1, 3, 100],
                    labels=["rare", "medium", "frequent"])

cmp = label_df[["label", "positives"]].copy()
cmp["bucket"] = new_bucket
for name in results:
    per = results[name][2]
    cmp[f"precision_{name}"] = per["precision"]
    cmp[f"recall_{name}"] = per["recall"]
    cmp[f"f1_{name}"] = per["f1"]

metric_cols = [c for c in cmp.columns if c.startswith(("precision_", "recall_", "f1_"))]
print("=== By bucket ===")
print(cmp.groupby("bucket", observed=True)[metric_cols].mean().round(3).T)

print("\n=== Per category (rarest first) ===")
print(cmp.sort_values("positives").round(3).to_string())

print("\nCategories with F1 = 0:")
for name in results:
    zeros = cmp.loc[cmp[f"f1_{name}"] == 0, "label"].tolist()
    print(f"  {name}: {len(zeros)} -> {zeros}")

cmp.to_csv("class_imbalance_per_category.csv", index=False)
print("\nSaved class_imbalance_per_category.csv")

=== By bucket ===
bucket                    rare  medium  frequent
precision_standard_bce   0.000   0.000     0.304
recall_standard_bce      0.000   0.000     0.335
f1_standard_bce          0.000   0.000     0.317
precision_weighted_sqrt  0.203   0.324     0.556
recall_weighted_sqrt     0.399   0.598     0.812
f1_weighted_sqrt         0.223   0.394     0.648

=== Per category (rarest first) ===
                                 label  positives    bucket  precision_standard_bce  recall_standard_bce  f1_standard_bce  precision_weighted_sqrt  recall_weighted_sqrt  f1_weighted_sqrt
30                  Price Restrictions         20      rare                   0.000                0.000            0.000                    0.000                 0.000             0.000
38   Unlimited/All-You-Can-Eat-License         22      rare                   0.000                0.000            0.000                    0.000                 0.000             0.000
34                  Source Code Escrow   

## Class Imbalance Optimization: Results

### Problem
Clause labels in the chunked CUAD training set are highly imbalanced:
- 61.8% of the 11,500 training chunks contain no clause at all.
- Every one of the 41 categories appears in under 5% of chunks; the rarest (Price Restrictions) appears in only 20 chunks (0.17%).
- 19 categories are "rare" (<1% of chunks), 13 "medium" (1–3%), 9 "frequent" (≥3%).

### Approach
Replaced standard binary cross-entropy with weighted BCE (`BCEWithLogitsLoss` with `pos_weight`).
- Weight per category = sqrt(negatives / positives), clipped to [1, 50].
- Square root dampens extreme ratios to keep training stable; resulting weights range from 4.6 (Parties) to 24.0 (Price Restrictions).
- Both models: DistilRoBERTa, 2 epochs, lr 2e-5, batch 16, max_length 512, seed 42, evaluated on the validation split at a fixed 0.5 threshold.

### Results

| Metric | Standard BCE | Weighted BCE |
|---|---|---|
| Micro-F1 | 0.256 | 0.491 |
| Macro-F1 | 0.070 | 0.370 |
| Categories with F1 = 0 | 37 / 41 | 7 / 41 |

| Bucket F1 | Standard BCE | Weighted BCE |
|---|---|---|
| Rare (<1%) | 0.000 | 0.223 |
| Medium (1–3%) | 0.000 | 0.394 |
| Frequent (≥3%) | 0.317 | 0.648 |

### Findings
1. **Standard BCE collapsed to predicting "no clause."** It only learned 4 short, formulaic categories (Parties, Agreement Date, Document Name, Effective Date) and scored 0 on the other 37.
2. **Weighted BCE recovered 30 of those 37 categories.** Macro-F1 rose from 0.07 to 0.37, with gains in every bucket, including frequent categories (0.32 → 0.65).
3. **Tradeoff: higher recall, lower precision.** The weighted model over-flags. For example, Affiliate License-Licensor reached 1.00 recall but 0.08 precision, and Governing Law reached 0.94 recall but 0.38 precision. For contract review, missing a clause is usually costlier than flagging an extra one, so this tradeoff favors weighted BCE.
4. **Small precision cost on categories that already worked.** Effective Date F1 dropped 0.59 → 0.55 and Agreement Date 0.74 → 0.73; Parties was unchanged.
5. **7 categories still score 0**, all among the rarest (20–63 training chunks): Price Restrictions, Unlimited/All-You-Can-Eat License, Source Code Escrow, Most Favored Nation, Third Party Beneficiary, Non-Disparagement, Joint IP Ownership.
6. **Some categories remain low-recall** even with weighting (Change of Control 0.17, Volume Restriction 0.17, Minimum Commitment 0.21), suggesting varied or ambiguous wording rather than just frequency.

### Limitations
- Only 2 epochs; standard BCE may partially improve with longer training.
- Single seed and a 51-contract validation set, so rare-category scores rest on very few examples and are noisy.
- Fixed 0.5 threshold; results will shift after threshold tuning.

### Recommendations
- Use weighted BCE (sqrt) as the default loss for the team's training experiments.
- Threshold tuning per category should recover much of the lost precision.
- For the 7 zero-F1 categories, try stronger weights (raw ratio), focal loss, more epochs, or oversampling chunks that contain them.
- Per-category metrics saved to `class_imbalance_per_category.csv`.